# Frequency-domain tools — Bode, margins, Nyquist, root locus, step response

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/learn/teaching/frequency_domain_tools.ipynb)

This notebook is the classical control chapter, read through the [minilink](https://github.com/alx87grd/minilink) API. Every tool follows the same recipe:

1. **Linearize** the plant about an operating point $(\bar x, \bar u)$ — the Jacobians $A = \partial f/\partial x$, $B = \partial f/\partial u$, $C = \partial h/\partial x$, $D = \partial h/\partial u$.
2. **Pick one channel**, an output component from an input component, so the model is a SISO state space $(A, b, c, d)$.
3. **Compute with linear algebra** on those four matrices: eigenvalues for poles, $c\,(j\omega I - A)^{-1} b + d$ for the frequency response, $\operatorname{eig}(A - b K c)$ for the root locus, one matrix exponential for the step response.

The API mirrors the recipe. Each verb reads `tool(x_bar, u_bar, ..., of=<output>, wrt=<input>)`, has a data twin and a `plot_` twin, and renders with `backend="matplotlib"` or `backend="plotly"`:

| what you want | data | plot |
| --- | --- | --- |
| poles, zeros, gain | `pzmap` | `plot_pzmap` |
| transfer function | `transfer_function` | — |
| frequency response | `bode`, `frequency_response` | `plot_bode` |
| gain and phase margins | `margins` | drawn on `plot_bode` |
| Nyquist contour | `nyquist` | `plot_nyquist` |
| closed-loop poles vs gain | `root_locus` | `plot_root_locus` |
| unit-step response | `step_response`, `step_info` | `plot_step_response` |

We use the pendulum hanging down as the plant, a PD law as the controller, and the inverted pendulum for the root-locus lesson.

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

In [ ]:
import numpy as np

from minilink import ImpedanceController, InvertedPendulum, Pendulum, TransferFunction
from minilink.analysis import step_info

## 1. From a nonlinear plant to a linear channel

The pendulum is $\;(m l^2 + I)\,\ddot\theta = \tau - m g l \sin\theta - d\,\dot\theta$, a nonlinear system with state $x = [\theta, \dot\theta]$, input $u = \tau$ and output $y = x$. Hanging down, $\bar x = [0, 0]$ and $\bar u = 0$ is an equilibrium. Linearizing there gives

$$\dot x = A x + B u, \qquad y = C x + D u,$$

and the channel from the torque to the angle is the transfer function

$$G(s) = c\,(sI - A)^{-1} b + d = \frac{k}{s^2 + 2\zeta\omega_n s + \omega_n^2}.$$

`linearize` returns the four matrices as an `LTISystem`; `pzmap` returns the poles (eigenvalues of $A$), the transmission zeros and the leading gain $k$; `transfer_function` packs them into a `TransferFunction` block.

In [ ]:
plant = Pendulum()
plant.params["d"] = 0.5  # a little damping, so the resonance is finite

x_bar = np.array([0.0, 0.0])  # hanging down, at rest

lin = plant.linearize(x_bar)
print("A =\n", np.round(lin.A(), 3))
print("B =\n", np.round(lin.B(), 3))

zeros, poles, gain = plant.pzmap(x_bar)  # channel: theta from tau (defaults)
print("poles:", np.round(poles, 3), " zeros:", zeros, " gain:", gain)

G = plant.transfer_function(x_bar)
print(G.name, "  num:", np.round(G.numerator, 3), " den:", np.round(G.denominator, 3))

The two poles at $-0.125 \pm 2.21j$ are the eigenvalues of $A$: a lightly damped oscillator with $\omega_n = \sqrt{4.905} \approx 2.21$ rad/s and $\zeta \approx 0.06$. The pole-zero map draws them in the $s$-plane, `x` for poles and `o` for zeros.

In [ ]:
plant.plot_pzmap(x_bar)

## 2. Frequency response and the Bode diagram

Drive the linear channel with $u = \sin\omega t$: in steady state the output is a sinusoid at the same frequency, scaled by $|G(j\omega)|$ and shifted by $\angle G(j\omega)$. The Bode diagram plots both against $\omega$ on a log axis, the magnitude in decibels, $20\log_{10}|G|$.

For the pendulum the magnitude is flat at low frequency (the static gain $k/\omega_n^2$), peaks at the resonance, then falls at $-40$ dB per decade; the phase goes from $0$ to $-180°$ through $-90°$ at $\omega_n$. `bode` returns the samples; `plot_bode` draws them. The automatic frequency grid runs one decade below the slowest pole or zero to one decade above the fastest.

In [ ]:
w, magnitude_db, phase_deg = plant.bode(x_bar, w=[0.5, 2.21, 10.0])
for wk, mk, pk in zip(w, magnitude_db, phase_deg):
    print(f"w = {wk:5.2f} rad/s   |G| = {mk:7.2f} dB   phase = {pk:8.2f} deg")

plant.plot_bode(x_bar, margins=False)

The same figure on the plotly backend is interactive: hover a point for $\omega$, $|G|$ and the phase. This is the backend to use in Colab.

In [ ]:
plant.plot_bode(x_bar, margins=False, backend="plotly")

## 3. The loop gain and the stability margins

Close the loop with the PD law of `ImpedanceController`,

$$u = K_p\,(r - \theta) - K_d\,\dot\theta \quad\Longrightarrow\quad C(s) = K_d\, s + K_p,$$

so the **loop gain** is $L(s) = C(s)\,G(s)$: the transfer function around the loop with the feedback path opened. The margins read how far $L$ is from the critical point $-1$:

- the **gain margin** is $-20\log_{10}|L(j\omega_{pc})|$ at the phase crossover $\angle L = -180°$: how much the gain can grow before instability;
- the **phase margin** is $180° + \angle L(j\omega_{gc})$ at the gain crossover $|L| = 1$: how much delay the loop tolerates.

Build $L$ as a `TransferFunction` block by multiplying the controller polynomial into the plant numerator. Every analysis verb is a method on the block, since an LTI system is its own linearization.

In [ ]:
def loop_gain(Kp, Kd):
    numerator = np.polymul([Kd, Kp], G.numerator)  # C(s) G(s)
    L = TransferFunction(numerator, G.denominator)
    L.name = f"L(s), Kp = {Kp}, Kd = {Kd}"
    return L


for Kp, Kd in ((5.0, 0.5), (20.0, 2.0), (50.0, 1.0)):
    m = loop_gain(Kp, Kd).margins()
    print(f"Kp = {Kp:4.0f}  Kd = {Kd:3.1f}:  phase margin = {m.phase_margin_deg:5.1f} deg "
          f"at {m.w_gain_crossover:4.2f} rad/s,  gain margin = {m.gain_margin_db}")

A second-order plant with a PD law never reaches $-180°$, so the gain margin is infinite: the loop stays stable for any gain, and the phase margin alone measures the damping of the closed loop. `plot_bode` draws the crossover frequencies and prints the margins in the corner.

In [ ]:
L = loop_gain(20.0, 2.0)
L.plot_bode()

The Nyquist diagram is the same response drawn as a curve in the complex plane, $L(j\omega)$ for $\omega$ from $0$ to $\infty$ (solid) and its mirror image for negative frequencies (dashed). The **Nyquist criterion** counts encirclements of $-1$: with a stable $L$, the closed loop is stable when the contour does not encircle the critical point. The distance from the contour to $-1$ is the margin seen geometrically.

In [ ]:
L.plot_nyquist()

## 4. Root locus

The closed-loop poles are the roots of $1 + K\,L(s) = 0$. The root locus draws them in the $s$-plane as the gain $K$ sweeps from $0$ (the open-loop poles, `x`) to $\infty$ (the open-loop zeros, `o`, and the asymptotes). minilink computes each point as $\operatorname{eig}(A - b K c)$ of the channel's state-space model and joins the branches.

On the PD-compensated pendulum the two branches leave the open-loop poles and bend toward the zero at $s = -K_p/K_d$: more gain moves the poles left and up, faster and better damped, which is the phase-margin story seen from the $s$-plane.

In [ ]:
L.plot_root_locus()

gains, roots = L.root_locus()
print("branches:", roots.shape[1], " gain sweep:", f"{gains[1]:.3g} .. {gains[-1]:.3g}")

The inverted pendulum makes the lesson sharper. About the upright, the two open-loop poles are real, $\pm 2.21$. Feeding back the angle alone, $u = -K\theta$, moves them together, they meet at the origin and leave along the imaginary axis: no gain stabilizes the pole. A zero at $s = -2$, the PD compensator $C(s) = s/2 + 1$, pulls both branches into the left half-plane once $K$ is large enough.

In [ ]:
inverted = InvertedPendulum()
upright = np.array([0.0, 0.0])

inverted.plot_root_locus(upright)  # angle feedback alone

G_up = inverted.transfer_function(upright)
L_pd = TransferFunction(np.polymul([0.5, 1.0], G_up.numerator), G_up.denominator)
L_pd.name = "inverted pendulum with a PD compensator"
L_pd.plot_root_locus()

gains, roots = L_pd.root_locus()
stable = np.all(roots.real < 0.0, axis=1)
print(f"the PD loop is stable above K = {gains[np.argmax(stable)]:.3g}")

## 5. Closing the loop in minilink and checking the prediction

Wire the actual controller block to the nonlinear plant with `@`. The closed-loop diagram is itself a system, so the same tools apply to it: `jacobian("f", "x")` is the closed-loop $A$ matrix, whose eigenvalues must be the roots of $1 + L(s) = 0$ found above.

In [ ]:
Kp, Kd = 20.0, 2.0
loop = ImpedanceController(Kp=Kp, Kd=Kd) @ plant  # r -> [ctl] -> u -> [pendulum] -> y

A_cl = loop.jacobian("f", "x")
print("closed-loop poles, from the diagram: ", np.round(np.linalg.eigvals(A_cl), 3))
print("closed-loop poles, roots of 1 + L(s):", np.round(np.roots(np.polyadd(L.denominator, L.numerator)), 3))

The step response of the closed loop, from the reference $r$ to the angle $\theta$, is the linear prediction of what the loop does. Note the steady-state value: proportional feedback on a plant with gravity stiffness leaves a static error, $\theta_\infty / r = K_p / (K_p + \omega_n^2 / k)$. `step_info` reads rise time, settling time and overshoot off the samples; the plot prints them.

In [ ]:
time, theta = loop.step_response(of=("y", 0), wrt="r")
info = step_info(time, theta)
print(f"steady state = {info.steady_state:.3f}   (Kp / (Kp + wn^2 / k) = {Kp / (Kp + 4.905 / 0.5):.3f})")
print(f"overshoot    = {info.overshoot:.1f} %   rise time = {info.rise_time:.2f} s   settling time = {info.settling_time:.2f} s")

loop.plot_step_response(of=("y", 0), wrt="r")

Finally the check that no linear tool can make: simulate the nonlinear loop. A reference of $0.3$ rad stays close to the linear regime, so the nonlinear angle should settle near $0.67 \times 0.3$; a reference of $2$ rad does not, and the difference is what the linearization leaves out.

In [ ]:
for r in (0.3, 2.0):
    loop.inputs["r"].set_nominal_value(np.array([r]))
    plant.x0 = np.zeros(2)
    traj = loop.compute_trajectory(tf=8.0, verbose=False)
    print(f"r = {r}:  nonlinear final angle = {traj.x[0, -1]:.3f}   linear prediction = {info.steady_state * r:.3f}")

loop.plot_trajectory()

## 6. Recap

| step | minilink | the math underneath |
| --- | --- | --- |
| linearize at $(\bar x, \bar u)$ | `plant.linearize(x_bar)` | $A, B, C, D$ by autodiff or finite differences |
| one channel | `of=("y", i)`, `wrt=("u", j)` | rows and columns of $C, D$ and $B$ |
| poles, zeros, gain | `plant.pzmap(x_bar)` | $\operatorname{eig}(A)$, the Rosenbrock pencil, the first Markov parameter |
| frequency response | `plant.bode(x_bar)` | $c\,(j\omega I - A)^{-1} b + d$ |
| margins | `L.margins()` | crossings of $0$ dB and $-180°$ |
| root locus | `L.root_locus()` | $\operatorname{eig}(A - b K c)$ over a gain sweep |
| step response | `loop.step_response(of=..., wrt=...)` | $x_{k+1} = e^{A\Delta t} x_k + \ldots$, one matrix exponential |

Every `plot_` twin takes `backend="matplotlib"` or `backend="plotly"`, and every verb is also a function in `minilink.analysis` for scripts: `bode(plant, x_bar)`, `root_locus(L)`, `margins(L)`.